# 🤖 Week 1 — How LLMs Actually Work
### A beginner's guide with live experiments

---

**By the end of this notebook you will understand:**
- What a token is and why it matters
- How an LLM predicts the next word (and why it sometimes gets it wrong)
- What system prompt and user prompt are
- How to talk to GPT-4o-mini AND Claude Haiku in the same notebook
- Why models give confident wrong answers (hallucination)
- How to fix stale answers using Tavily live search

**No prior AI experience needed. Every cell is explained before you run it.**

---

### 📦 What we need
| Library | What it does |
|---|---|
| `openai` | Talks to GPT models |
| `anthropic` | Talks to Claude models |
| `tiktoken` | Counts tokens (OpenAI's tokeniser) |
| `tavily-python` | Live web search — gives models fresh facts |
| `python-dotenv` | Loads API keys from a `.env` file |

---
## ⚙️ Cell 0 — Install & Setup
Run this first. It installs everything and loads your API keys.

In [ ]:
# Install all libraries we need
!pip install openai anthropic tiktoken tavily-python python-dotenv --quiet

import os
from dotenv import load_dotenv

# Load keys from .env file in the same folder
# Your .env file should look like this:
#   OPENAI_API_KEY=sk-...
#   ANTHROPIC_API_KEY=sk-ant-...
#   TAVILY_API_KEY=tvly-...
load_dotenv()

OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY",    "")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
TAVILY_API_KEY    = os.getenv("TAVILY_API_KEY",    "")

# Check which keys are loaded
print("API Key Status:")
print(f"  OpenAI   : {'✅ loaded' if OPENAI_API_KEY    else '❌ missing — add to .env'}")
print(f"  Anthropic: {'✅ loaded' if ANTHROPIC_API_KEY else '❌ missing — add to .env'}")
print(f"  Tavily   : {'✅ loaded' if TAVILY_API_KEY    else '❌ missing — get free key at tavily.com'}")

# Create clients
from openai import OpenAI
import anthropic

oai_client    = OpenAI(api_key=OPENAI_API_KEY)
claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print()
print("✅  All clients ready!")

---
## 🧩 Section 1 — What Is a Token?

An LLM does **not** read words. It reads **tokens** — chunks of text that can be:
- A whole word: `cat` → 1 token
- Part of a word: `unbelievable` → `un` + `believ` + `able` → 3 tokens
- A punctuation mark or space

**Why does this matter?**
- You pay per token (not per word)
- The model has a token limit (context window)
- Some tasks are hard for models *because* of tokenisation (like counting letters)

Let's visualise this 👇

In [ ]:
import tiktoken

# tiktoken is OpenAI's tokeniser — let's use it to see exactly what the model reads
enc = tiktoken.encoding_for_model("gpt-4o-mini")

def show_tokens(text):
    """Break text into tokens and display them visually."""
    tokens  = enc.encode(text)
    decoded = [enc.decode([t]) for t in tokens]

    print(f"📝 Input    : {text!r}")
    print(f"🔢 Tokens   : {decoded}")
    print(f"📊 Count    : {len(tokens)} tokens | {len(text)} characters")
    print(f"💰 Cost est : ${len(tokens) * 0.00000015:.8f}  (at gpt-4o-mini input rate)")
    print()

# ── Try different types of text ───────────────────────────────────────────────
print("=" * 60)
print("TOKEN VISUALISER")
print("=" * 60)
print()

show_tokens("Hello")
show_tokens("unbelievable")
show_tokens("ServiceNow incident management")
show_tokens("नमस्ते")                            # Hindi — notice MORE tokens than English
show_tokens("2024-11-07T14:30:00Z")              # Timestamp — splits unexpectedly
show_tokens("strawberry")                         # We'll use this in the next cell!

print("💡 Notice: Hindi uses far more tokens than English for the same meaning.")
print("   Non-English users pay more AND hit context limits faster.")

In [ ]:
# ── The classic trick question ────────────────────────────────────────────────
# Ask the model to count letters. Because it sees TOKENS not characters,
# it will get some of these wrong.

print("=" * 60)
print("THE CHARACTER COUNTING TRAP")
print("=" * 60)
print()
print("Asking GPT-4o-mini to count letters...")
print("(It reads tokens, not characters — watch what happens)")
print()

questions = [
    ("How many letter r's are in the word 'strawberry'?",  "3"),
    ("What is the 7th character in 'unbelievable'?",       "i"),
    ("How many characters in the word 'Mississippi'?",     "11"),
]

for question, correct_answer in questions:
    response = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}],
        temperature=0
    )
    model_answer = response.choices[0].message.content.strip()

    # Check if the model got it right
    is_correct = correct_answer.lower() in model_answer.lower()
    icon = "✅ Correct" if is_correct else "❌ Wrong  "

    print(f"Q: {question}")
    print(f"   Model says : {model_answer[:80]}")
    print(f"   Truth      : {correct_answer}")
    print(f"   Result     : {icon}")
    print()

print("🔑 KEY LESSON: For counting, parsing, or formatting tasks — use Python code,")
print("   not an LLM. LLMs are pattern matchers, not calculators.")

---
## 🎲 Section 2 — How LLMs Predict the Next Word

An LLM doesn't "know" the answer. It **predicts the most likely next token**, over and over,
until it decides to stop.

Think of it like autocomplete on your phone — but trained on most of the internet.

```
You type:   "The capital of France is ..."
Model sees: What token most likely follows this sequence?
Answer:     "Paris" (very high probability)
            "Lyon"  (low probability)
            "blue"  (almost zero probability)
```

The **temperature** setting controls how adventurous the model is:
- `temperature=0` → always picks the most likely token (deterministic)
- `temperature=1` → samples proportionally from likely tokens (creative)
- `temperature=2` → samples from even unlikely tokens (chaotic)

In [ ]:
# ── Visualise "next word prediction" with logprobs ────────────────────────────
# OpenAI lets us see the top 5 tokens it considered at each step.
# This makes the prediction process VISIBLE.

import math

def show_next_word_probabilities(prompt, n_tokens=5):
    """
    Complete the prompt and show the top 5 candidate tokens
    the model considered at the FIRST prediction step.
    """
    response = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1,           # only predict ONE token
        logprobs=True,          # show probabilities
        top_logprobs=5,         # show top 5 candidates
        temperature=1
    )

    top5 = response.choices[0].logprobs.content[0].top_logprobs

    print(f"📝 Prompt: \"{prompt}\"")
    print(f"\n   Top {len(top5)} candidates for the NEXT word:")
    print(f"   {'Token':<20} {'Probability':>12} {'Bar'}")
    print(f"   {'─'*20} {'─'*12} {'─'*25}")

    for lp in top5:
        prob    = math.exp(lp.logprob) * 100      # convert log-prob to %
        bar     = "█" * int(prob / 2)              # visual bar
        token   = repr(lp.token)                   # show spaces and newlines clearly
        print(f"   {token:<20} {prob:>11.1f}%  {bar}")
    print()


print("=" * 60)
print("NEXT WORD PROBABILITY VISUALISER")
print("=" * 60)
print()

# Unambiguous — one clear answer
show_next_word_probabilities("The capital of France is")

# Slightly ambiguous
show_next_word_probabilities("Python is a programming")

# Very ambiguous — model has to guess
show_next_word_probabilities("The best way to fix a broken")

print("🔑 KEY LESSON:")
print("   When the model is CERTAIN → one token has ~99% probability (Paris)")
print("   When the model is UNCERTAIN → probability spreads across many tokens")
print("   Hallucination happens when the model picks a confident-sounding token")
print("   even when it has no real knowledge to back it up.")

In [ ]:
# ── Temperature experiment — CORRECTLY designed ───────────────────────────────
# KEY INSIGHT: temperature only shows visible variation on AMBIGUOUS or CREATIVE tasks.
# On clear factual questions ("capital of France?") the model is already 99% sure
# of the answer, so temperature makes no visible difference.

print("=" * 60)
print("TEMPERATURE EXPERIMENT")
print("=" * 60)
print()

# ── Experiment A: CREATIVE task — tagline writing ────────────────────────────
print("── Experiment A: Creative task (tagline writing)")
print("   temp=0 → same answer every time (useless for brainstorming)")
print("   temp=1 → different options each run (great for ideation)")
print()

prompt_creative = "Write one short, catchy tagline for an AI-powered IT helpdesk. Max 10 words."

print("   temp=0 (5 runs):")
seen = set()
for i in range(5):
    r = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_creative}],
        temperature=0, max_tokens=30
    )
    ans = r.choices[0].message.content.strip()
    seen.add(ans)
    print(f"     Run {i+1}: {ans}")
print(f"   → Unique answers: {len(seen)}/5")
print()

print("   temp=1.0 (5 runs):")
seen2 = set()
for i in range(5):
    r = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_creative}],
        temperature=1.0, max_tokens=30
    )
    ans = r.choices[0].message.content.strip()
    seen2.add(ans)
    print(f"     Run {i+1}: {ans}")
print(f"   → Unique answers: {len(seen2)}/5")
print()

# ── Experiment B: AMBIGUOUS classification ───────────────────────────────────
print("── Experiment B: Ambiguous classification task")
print("   This ticket could be Network, Software, OR Access — model is uncertain")
print()

ambiguous_ticket = """
After the Windows 11 update, some staff cannot reach SharePoint via office Wi-Fi,
but it works fine on mobile data. Could be DNS, proxy config, or a permissions issue.
"""

system = "Classify into ONE category: [Network, Hardware, Software, Access, Other]. Return only the category name."

print("   temp=0 (5 runs):")
r0 = []
for i in range(5):
    r = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"system","content":system},{"role":"user","content":ambiguous_ticket}],
        temperature=0, max_tokens=10
    )
    ans = r.choices[0].message.content.strip()
    r0.append(ans)
    print(f"     Run {i+1}: {ans}")
print(f"   → Unique: {set(r0)}")
print()

print("   temp=1.5 (5 runs):")
r1 = []
for i in range(5):
    r = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"system","content":system},{"role":"user","content":ambiguous_ticket}],
        temperature=1.5, max_tokens=10
    )
    ans = r.choices[0].message.content.strip()
    r1.append(ans)
    print(f"     Run {i+1}: {ans}")
print(f"   → Unique: {set(r1)}")
print()
print("🔑 RULE: temp=0 for decisions/routing. temp=0.7-1.0 for drafting/brainstorming.")

---
## 💬 Section 3 — System Prompt vs User Prompt

Every LLM conversation has two layers:

```
┌─────────────────────────────────────────────┐
│  SYSTEM PROMPT  (the instructions)           │
│  → Set by the developer / trainer            │
│  → Defines role, tone, rules, format         │
│  → User usually cannot see this              │
├─────────────────────────────────────────────┤
│  USER PROMPT  (the request)                  │
│  → Typed by the user                         │
│  → The actual question or task               │
└─────────────────────────────────────────────┘
```

**The system prompt is like a job description given to a new employee before they start.**  
The user prompt is what the customer then asks that employee.

In [ ]:
# ── Same question, 3 different system prompts ─────────────────────────────────
# Watch how the SAME user question gets completely different answers
# based purely on the system prompt.

def ask(system_prompt, user_prompt, model="gpt-4o-mini", temperature=0.7):
    """Simple wrapper — send system + user prompt, get response back."""
    response = oai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=200
    )
    return response.choices[0].message.content.strip()


# Same user question every time
USER_QUESTION = "My laptop is running very slow today."

print("=" * 60)
print("SYSTEM PROMPT SHAPES THE ANSWER")
print(f"User question: \"{USER_QUESTION}\"")
print("=" * 60)
print()

personas = [
    (
        "Friendly IT Helpdesk",
        "You are a friendly IT helpdesk assistant. Be warm, empathetic, and give 2-3 practical steps. Use simple language."
    ),
    (
        "Senior Security Engineer",
        "You are a cybersecurity expert. Always consider security implications first. Be technical and precise."
    ),
    (
        "Cost-Focused IT Manager",
        "You are an IT manager focused on cost efficiency. Prioritise free solutions and self-service before escalation. Be brief."
    ),
]

for name, system_prompt in personas:
    response = ask(system_prompt, USER_QUESTION)
    print(f"── 🎭 Persona: {name} ──")
    print(f"   System: \"{system_prompt[:70]}...\"")
    print()
    print(f"   Response:")
    for line in response.splitlines():
        print(f"   {line}")
    print()

print("🔑 KEY LESSON: The system prompt is your most powerful tool.")
print("   It defines who the model IS, not just what to say.")

In [ ]:
# ── Effect of system prompt on OUTPUT FORMAT ──────────────────────────────────
# System prompt can control format just as powerfully as content.

print("=" * 60)
print("SYSTEM PROMPT CONTROLS OUTPUT FORMAT")
print("=" * 60)
print()

incident = """
At 2:30 PM the entire finance floor lost access to SAP.
About 80 users affected. Root cause was an expired SSL certificate
on the load balancer. Fixed by 3:15 PM. Downtime: 45 minutes.
"""

format_prompts = [
    ("Bullet Points",
     "Summarise IT incidents as exactly 3 bullet points. Start each with •"),

    ("One Sentence",
     "Summarise IT incidents in exactly ONE sentence. Max 25 words."),

    ("JSON",
     'Summarise IT incidents as JSON: {"impact": str, "cause": str, "resolution": str, "downtime_mins": int}. Return only valid JSON.'),
]

for format_name, system_prompt in format_prompts:
    response = ask(system_prompt, incident, temperature=0)
    print(f"── Format: {format_name} ──")
    print(f"   System says: \"{system_prompt[:60]}...\"")
    print(f"   Output:")
    for line in response.splitlines():
        print(f"     {line}")
    print()

---
## 🤖 Section 4 — Two Models, Same Question

You have access to **two different AI companies** in this notebook:

| Model | Company | Good at | Cost |
|---|---|---|---|
| `gpt-4o-mini` | OpenAI | Fast, cheap, reliable | Very low |
| `claude-haiku-4-5` | Anthropic | Careful, nuanced, refuses harmful requests | Very low |

They are trained differently, have different training data cutoffs,
and make different mistakes. **Never assume two models agreeing = correct.**

In [ ]:
# ── Helper: ask Claude ────────────────────────────────────────────────────────
def ask_claude(system_prompt, user_prompt, temperature=0.7):
    """Send a message to Claude Haiku and return the response."""
    response = claude_client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=300,
        temperature=temperature,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}]
    )
    return response.content[0].text.strip()


# ── Compare GPT and Claude on the same tasks ──────────────────────────────────
print("=" * 60)
print("GPT-4o-mini  vs  Claude Haiku")
print("=" * 60)
print()

system = "You are an IT helpdesk assistant. Be helpful and concise."

comparison_tasks = [
    "In one sentence: what is the difference between an incident and a problem in ITIL?",
    "My VPN keeps disconnecting every 30 minutes. What are the 3 most likely causes?",
    "Should we store user passwords in our database in plain text? Answer in one word, then explain why.",
]

for task in comparison_tasks:
    gpt_ans   = ask(system, task, temperature=0)
    claude_ans = ask_claude(system, task, temperature=0)

    print(f"❓ Question: {task}")
    print()
    print(f"   🟢 GPT-4o-mini:")
    for line in gpt_ans.splitlines():
        print(f"      {line}")
    print()
    print(f"   🟣 Claude Haiku:")
    for line in claude_ans.splitlines():
        print(f"      {line}")
    print()
    print("   " + "─" * 55)
    print()

In [ ]:
# ── Where models disagree — and BOTH sound confident ─────────────────────────
# This is the most important demo in Week 1.
# Two models can give different answers and both sound 100% sure.

print("=" * 60)
print("DANGER ZONE: Both models sound confident")
print("but they DISAGREE — at least one is wrong")
print("=" * 60)
print()

factual_questions = [
    "What was ServiceNow's total revenue in fiscal year 2023?",
    "Who is the current CEO of Infosys?",
    "What is your exact training data cutoff date?",
]

system_factual = "Answer factually and precisely. Be direct."

for q in factual_questions:
    gpt_ans   = ask(system_factual, q, temperature=0)
    claude_ans = ask_claude(system_factual, q, temperature=0)

    agree = gpt_ans.strip()[:40].lower() == claude_ans.strip()[:40].lower()

    print(f"❓ {q}")
    print(f"   🟢 GPT   : {gpt_ans[:100]}")
    print(f"   🟣 Claude: {claude_ans[:100]}")
    print(f"   🤝 Agree : {'Likely yes' if agree else '⚠️  THEY DISAGREE — verify with a primary source!'}")
    print()

print("🔑 KEY LESSON:")
print("   Model agreement does NOT equal truth.")
print("   For facts that matter — always verify with a primary source.")
print("   In Section 6 we'll use Tavily to search the web for live, accurate data.")

---
## 🌀 Section 5 — Hallucination: Why Models Make Things Up

Hallucination is not a bug — it's a **structural consequence** of how LLMs work.

The model is always predicting the next most likely token. When it doesn't know something,
it predicts what a **plausible answer would look like** — and delivers that with full confidence.

Three types of hallucination you'll see in production:

| Type | Example |
|---|---|
| **Fabrication** | Inventing statistics, dates, or people that don't exist |
| **Outdated fact** | Stating something that was true 2 years ago as current |
| **Conflation** | Mixing facts from two similar things (two CEOs, two products) |

In [ ]:
# ── Hallucination Type 1: Fabricated citations ────────────────────────────────
# Ask for academic citations. The model will INVENT plausible-looking ones.
# This is one of the most dangerous patterns in professional settings.

print("=" * 60)
print("HALLUCINATION TYPE 1 — Fabricated Citations")
print("=" * 60)
print()

citation_prompt = """
Give me 3 academic papers on 'ROI of IT automation in enterprise'.
Include: author, title, journal, year, and DOI.
"""

print("GPT-4o-mini says:")
print()
response = ask("You are a research assistant.", citation_prompt, temperature=0)
print(response)
print()
print("⚠️  LIVE DEMO CHALLENGE:")
print("   Copy the first DOI above → go to doi.org → does it exist?")
print("   In most runs, at least 1 of 3 citations will lead to a 404 or wrong paper.")
print("   The author names, journal names, and years all sound completely real.")

In [ ]:
# ── Hallucination Type 2: Outdated facts ──────────────────────────────────────
# Models have a training cutoff. Anything that changed after that date
# will be stated confidently but wrongly.

print("=" * 60)
print("HALLUCINATION TYPE 2 — Outdated Facts")
print("=" * 60)
print()

fast_changing_questions = [
    "What is the latest version of Python?",
    "Who is the current CEO of OpenAI?",
    "What is the current USD to INR exchange rate?",
]

system_direct = "Answer directly and specifically. Give exact numbers and names."

for q in fast_changing_questions:
    gpt_ans = ask(system_direct, q, temperature=0)
    print(f"Q: {q}")
    print(f"   GPT says: {gpt_ans[:120]}")
    print(f"   ⚠️  This may be out of date — the model has a training cutoff!")
    print()

print("We'll fix this in Section 6 using live web search.")

In [ ]:
# ── Hallucination Type 3: Conflation ─────────────────────────────────────────
# The model blends facts from two similar things.
# Very common with similar products, people with similar names, or related frameworks.

print("=" * 60)
print("HALLUCINATION TYPE 3 — Conflation")
print("=" * 60)
print()

conflation_q = """
Answer each question precisely:
1. Which IT framework introduced the 'Service Value Chain' concept?
2. Which framework has exactly 40 governance and management objectives?
3. In which year was ITIL 4 released?
"""

gpt_ans   = ask("Answer factually.", conflation_q, temperature=0)
claude_ans = ask_claude("Answer factually.", conflation_q, temperature=0)

print("🟢 GPT-4o-mini:")
print(gpt_ans)
print()
print("🟣 Claude Haiku:")
print(claude_ans)
print()
print("Ground truth:")
print("  Q1: ITIL 4 introduced the Service Value Chain")
print("  Q2: COBIT 2019 has 40 governance and management objectives")
print("  Q3: ITIL 4 was released in 2019")
print()
print("🔑 Do both models get all 3 right? Did either conflate ITIL and COBIT?")

---
## 🔍 Section 6 — Fix Stale Answers with Tavily Live Search

**The problem:** LLMs have a training cutoff. Anything that changed after that date will be wrong.

**The fix:** Before answering, search the web for fresh data and inject it into the prompt.
This is called **Retrieval-Augmented Generation (RAG)** — we'll do a full deep-dive in Week 3.

Today we use **Tavily** — a search API built specifically for LLMs.

> 🆓 **Free tier:** 1,000 searches/month at [tavily.com](https://tavily.com) — no credit card needed.

```
WITHOUT TAVILY:                 WITH TAVILY:
User asks question              User asks question
       ↓                               ↓
LLM answers from                Tavily searches web → gets fresh results
training data                          ↓
(may be outdated)               Fresh results injected into prompt
                                       ↓
                                LLM answers from fresh data
                                (grounded, accurate)
```

In [ ]:
# ── Tavily setup ──────────────────────────────────────────────────────────────
from tavily import TavilyClient

if not TAVILY_API_KEY:
    print("❌ TAVILY_API_KEY not found in .env")
    print("   Get a free key at: https://tavily.com")
    print("   Add to .env: TAVILY_API_KEY=tvly-...")
else:
    tavily = TavilyClient(api_key=TAVILY_API_KEY)
    print("✅ Tavily client ready")

    # Test with a simple search
    print()
    print("Quick test — searching for 'latest Python version'...")
    results = tavily.search("latest Python version 2024", max_results=2)
    for r in results["results"]:
        print(f"  📰 {r['title']}")
        print(f"     {r['content'][:120]}...")
        print()

In [ ]:
# ── Side-by-side: LLM alone vs LLM + Tavily ──────────────────────────────────
# The questions from Section 5 that the model got wrong or outdated.

def search_and_ask(question, search_query=None):
    """
    1. Search Tavily for fresh results
    2. Inject results into the prompt
    3. Ask GPT to answer using only the fresh results
    """
    query = search_query or question

    # Step 1: Search
    results  = tavily.search(query, max_results=3)
    snippets = "\n\n".join([
        f"Source: {r['url']}\n{r['content'][:300]}"
        for r in results["results"]
    ])

    # Step 2: Build augmented prompt
    augmented_prompt = f"""
Use ONLY the search results below to answer the question.
If the results don't contain the answer, say so.
Cite the source URL after your answer.

SEARCH RESULTS:
{snippets}

QUESTION: {question}
"""

    # Step 3: Ask GPT
    answer = ask("You are a factual assistant. Use only the provided search results.",
                 augmented_prompt, temperature=0)
    return answer, snippets


if TAVILY_API_KEY:
    print("=" * 60)
    print("LLM ALONE  vs  LLM + TAVILY LIVE SEARCH")
    print("=" * 60)
    print()

    questions_to_fix = [
        ("What is the latest stable version of Python?", "Python latest stable release"),
        ("Who is the current CEO of OpenAI?",            "OpenAI CEO 2024 current"),
    ]

    for question, search_query in questions_to_fix:
        print(f"❓ {question}")
        print()

        # LLM alone
        llm_only = ask("Answer directly.", question, temperature=0)
        print(f"   ❌ LLM alone (may be outdated):")
        print(f"      {llm_only[:150]}")
        print()

        # LLM + Tavily
        llm_grounded, _ = search_and_ask(question, search_query)
        print(f"   ✅ LLM + Tavily (fresh from the web):")
        for line in llm_grounded.splitlines():
            print(f"      {line}")
        print()
        print("   " + "─" * 50)
        print()
else:
    print("⚠️  Skipping Tavily demo — add TAVILY_API_KEY to your .env file")
    print("   Get a free key at https://tavily.com")

In [ ]:
# ── IT-specific live search demo ──────────────────────────────────────────────
# Use case relevant to IT teams: check if a software version has known vulnerabilities

if TAVILY_API_KEY:
    print("=" * 60)
    print("PRACTICAL IT USE CASE — Live vulnerability check")
    print("=" * 60)
    print()

    vuln_question = "Are there any known critical vulnerabilities in Windows 11 23H2 released in 2024?"

    print(f"Question: {vuln_question}")
    print()

    print("🔍 Searching Tavily...")
    vuln_answer, snippets = search_and_ask(
        vuln_question,
        "Windows 11 23H2 critical vulnerabilities CVE 2024"
    )

    print("📋 Grounded answer:")
    print(vuln_answer)
    print()
    print("💡 Without Tavily, the model would only know about vulnerabilities")
    print("   that existed before its training cutoff — missing anything recent.")
else:
    print("⚠️  Add TAVILY_API_KEY to your .env to run this demo")

---
## 🎛️ Section 7 — Build Your Own: Interactive Prompt Lab

Now it's your turn. Modify the cells below to experiment.

**Three exercises — try at least one:**

In [ ]:
# ── Exercise A: Change the persona ───────────────────────────────────────────
# Try different system prompts and see how the answer changes.
# Ideas: "pirate", "Yoda", "grumpy senior developer", "5-year-old"

MY_SYSTEM_PROMPT = """
You are a friendly IT assistant who explains everything using cooking analogies.
Keep it simple and fun.
"""

MY_QUESTION = "What is a firewall and why do we need one?"

# ── Try with GPT ─────────────────────────────────────────────────────────────
gpt_response = ask(MY_SYSTEM_PROMPT, MY_QUESTION, temperature=0.7)
print("🟢 GPT-4o-mini says:")
print(gpt_response)
print()

# ── Try the same with Claude ──────────────────────────────────────────────────
claude_response = ask_claude(MY_SYSTEM_PROMPT, MY_QUESTION, temperature=0.7)
print("🟣 Claude Haiku says:")
print(claude_response)

In [ ]:
# ── Exercise B: Format control ────────────────────────────────────────────────
# The system prompt below asks for structured output.
# Change the format requirements and run again.

FORMAT_SYSTEM = """
You are an IT incident analyst.
Always respond in this exact format:

SEVERITY: [P1/P2/P3/P4]
CATEGORY: [one word]
AFFECTED: [number of users]
ACTION:   [one sentence — what to do right now]
"""

incidents_to_classify = [
    "The payment gateway has been returning errors for 20 minutes. 300+ customers affected.",
    "My mouse cursor sometimes jumps across the screen.",
    "The entire Bangalore office Wi-Fi is down. 500 staff cannot work.",
]

print("=" * 60)
print("STRUCTURED OUTPUT DEMO")
print("=" * 60)

for incident in incidents_to_classify:
    response = ask(FORMAT_SYSTEM, incident, temperature=0)
    print(f"\nIncident: {incident[:70]}..." if len(incident) > 70 else f"\nIncident: {incident}")
    print(response)
    print()

In [ ]:
# ── Exercise C: Your own Tavily search ───────────────────────────────────────
# Ask something you actually want to know the current answer to.

MY_LIVE_QUESTION   = "What are the latest AI features announced by ServiceNow in 2024?"
MY_SEARCH_QUERY    = "ServiceNow AI features announcements 2024"   # used for Tavily search

if TAVILY_API_KEY:
    print(f"🔍 Searching for: {MY_SEARCH_QUERY}")
    print()
    answer, _ = search_and_ask(MY_LIVE_QUESTION, MY_SEARCH_QUERY)
    print("Answer (grounded in live web results):")
    print(answer)
else:
    print("⚠️  Add TAVILY_API_KEY to your .env to use live search")
    # Fallback: answer from model memory only
    answer = ask("Answer helpfully.", MY_LIVE_QUESTION, temperature=0)
    print("Answer (from model memory — may be outdated):")
    print(answer)

---
## 📊 Section 8 — Session Summary
See how many tokens and API calls you used in this session.

In [ ]:
# ── Token & cost summary for this notebook session ────────────────────────────
# This uses the shared LLMClient + Tracer from utils.py

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from shared.utils import LLMClient, Tracer, banner

# All calls we made above went through the raw OpenAI/Anthropic clients.
# From Week 2 onwards we use LLMClient which tracks everything automatically.
# Here we show a simple manual estimate instead.

banner("📊 What you used in this session")

print("Approximate usage in this notebook:")
print()

# Rough estimates based on typical runs
estimates = [
    ("Token visualiser (Section 1)",          "gpt-4o-mini", 250,  80),
    ("Character counting (Section 1)",        "gpt-4o-mini", 300, 150),
    ("Next-word probs (Section 2)",           "gpt-4o-mini", 200,  15),
    ("Temperature experiments (Section 2)",   "gpt-4o-mini", 800, 200),
    ("System prompt personas (Section 3)",    "gpt-4o-mini", 600, 450),
    ("GPT vs Claude comparison (Section 4)",  "both",       1200, 600),
    ("Hallucination demos (Section 5)",       "both",        800, 400),
    ("Tavily + grounded answers (Section 6)", "gpt-4o-mini", 900, 300),
]

total_input  = sum(e[2] for e in estimates)
total_output = sum(e[3] for e in estimates)

# gpt-4o-mini: $0.00015/1K input, $0.00060/1K output
est_cost = (total_input * 0.00015 + total_output * 0.00060) / 1000

print(f"  {'Section':<45} {'~Input':>8} {'~Output':>8}")
print("  " + "─" * 65)
for name, model, inp, out in estimates:
    print(f"  {name:<45} {inp:>8,} {out:>8,}")
print("  " + "─" * 65)
print(f"  {'TOTAL':<45} {total_input:>8,} {total_output:>8,}")
print()
print(f"  💰 Estimated cost: ~${est_cost:.5f} USD  (less than a fraction of a rupee)")
print()
print("  From Week 2 onwards, the shared LLMClient tracks this automatically.")

---
## ✅ Week 1 — Key Takeaways

| What we learned | Why it matters for your work |
|---|---|
| Models read **tokens**, not words | Character tasks must use code, not LLMs. Non-English = more tokens = more cost |
| LLMs **predict the next token** probabilistically | They're not databases — they're very sophisticated autocomplete |
| **Temperature** controls randomness | temp=0 for decisions and routing. High temp for brainstorming |
| **System prompt** shapes everything | Role, tone, format, rules — set once, affects every response |
| **GPT and Claude** are different | Same question can get different answers. Neither is always right |
| Hallucination has **3 patterns** | Fabrication, stale data, conflation — all sound equally confident |
| **Tavily** gives models live data | Fixes the training cutoff problem — this is the foundation of RAG (Week 3) |

---

### 🚀 Coming up in Week 2
We'll learn to **control the model's output reliably**:
- Why naive prompts break in production
- Few-shot examples that teach business logic
- JSON output that your systems can parse
- Prompt injection attacks — and how to defend against them

---

### 📝 Before next week
Try this on your own:
1. Pick a real question from your work domain
2. Ask it to GPT and Claude — do they agree?
3. If it's a fast-changing fact, use the Tavily cell (Exercise C) to get a live answer
4. Note: did the model admit uncertainty, or sound confident regardless?